In [3]:
!pip install ultralytics roboflow pyyaml -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import shutil
import yaml
from pathlib import Path

from roboflow import Roboflow
from ultralytics import YOLO

old_model_path = "/kaggle/input/datasets/leslyhihihi/googlecolab2/best.pt"

if os.path.exists(old_model_path):
    print("Старая модель найдена:", old_model_path)
else:
    raise FileNotFoundError("Старая модель не найдена")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Старая модель найдена: /kaggle/input/datasets/leslyhihihi/googlecolab2/best.pt


In [3]:
rf = Roboflow(api_key="Ylzy6XETZO8Q6xhYsyO5")

datasets = []

# 1. Сумки
project = rf.workspace("rts-mlsjs").project("bag-detection-jhlby-a0bal")
datasets.append(project.version(1).download("yolov8"))

# 2. Еще сумки
project = rf.workspace("rts-mlsjs").project("bags-sapbe-dwyfz")
datasets.append(project.version(1).download("yolov8"))

# 3. Люди и сумки
project = rf.workspace("rts-mlsjs").project("people-and-bags-u3zuh")
datasets.append(project.version(1).download("yolov8"))

# 4. Handbag
project = rf.workspace("rts-mlsjs").project("handbag-zgbci-mcy84")
datasets.append(project.version(1).download("yolov8"))

# 5. Backpack
project = rf.workspace("rts-mlsjs").project("backpack-qkkdx-zzycl")
datasets.append(project.version(1).download("yolov8"))

print("Все датасеты скачаны")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bag-detection-1 in yolov8:: 100%|██████████| 13015/13015 [00:03<00:00, 4090.36it/s] 


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Bags-1 in yolov8:: 100%|██████████| 9215/9215 [00:00<00:00, 10040.32it/s]

loading Roboflow workspace...


...


Extracting Dataset Version Zip to Handbag-1 in yolov8:: 100%|██████████| 3401/3401 [00:00<00:00, 5119.12it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to Backpack-1 in yolov8:: 100%|██████████| 4619/4619 [00:00<00:00, 8471.78it/s]

Все датасеты скачаны


In [4]:
dataset_paths = []

for root, dirs, files in os.walk("/kaggle/working"):
    if "data.yaml" in files:
        dataset_paths.append(root)

print("Найденные датасеты:")
for path in dataset_paths:
    print(path)

Найденные датасеты:
/kaggle/working/people-and-bags-1
/kaggle/working/Bags-1
/kaggle/working/bag-detection-1
/kaggle/working/Backpack-1
/kaggle/working/Handbag-1


In [5]:
final_path = Path("/kaggle/working/final_people_bag_dataset_v3")

for split in ["train", "valid", "test"]:
    (final_path / split / "images").mkdir(parents=True, exist_ok=True)
    (final_path / split / "labels").mkdir(parents=True, exist_ok=True)

target_names = {
    # люди
    "person": 0,
    "people": 0,
    "human": 0,
    "man": 0,
    "woman": 0,
    "person with bag": 0,
    "person without bag": 0,

    # сумки
    "bag": 1,
    "bags": 1,
    "handbag": 1,
    "hand bag": 1,
    "backpack": 1,
    "back pack": 1,
    "luggage": 1,
    "suitcase": 1,
    "purse": 1,
    "shoulder bag": 1,
    "school bag": 1
}


def normalize_name(name):
    return str(name).lower().strip().replace("_", " ").replace("-", " ")


def get_class_map(data_yaml_path):
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)

    names = data["names"]

    if isinstance(names, dict):
        names = [names[i] for i in sorted(names.keys())]

    class_map = {}

    for old_id, name in enumerate(names):
        clean_name = normalize_name(name)

        if clean_name in target_names:
            class_map[old_id] = target_names[clean_name]

    return class_map, names


def copy_dataset(dataset_dir, prefix):
    dataset_dir = Path(dataset_dir)
    data_yaml = dataset_dir / "data.yaml"

    class_map, old_names = get_class_map(data_yaml)

    print("Датасет:", dataset_dir)
    print("Старые классы:", old_names)
    print("Карта классов:", class_map)
    print("-" * 60)

    for split in ["train", "valid", "test"]:
        img_dir = dataset_dir / split / "images"
        lbl_dir = dataset_dir / split / "labels"

        if not img_dir.exists() or not lbl_dir.exists():
            continue

        for img_path in img_dir.glob("*"):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp"]:
                continue

            label_path = lbl_dir / f"{img_path.stem}.txt"

            if not label_path.exists():
                continue

            new_label_lines = []

            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()

                if len(parts) < 5:
                    continue

                old_class_id = int(float(parts[0]))

                if old_class_id not in class_map:
                    continue

                new_class_id = class_map[old_class_id]
                parts[0] = str(new_class_id)

                new_label_lines.append(" ".join(parts))

            if len(new_label_lines) == 0:
                continue

            new_img_name = f"{prefix}_{split}_{img_path.name}"
            new_lbl_name = f"{prefix}_{split}_{img_path.stem}.txt"

            shutil.copy(img_path, final_path / split / "images" / new_img_name)

            with open(final_path / split / "labels" / new_lbl_name, "w") as f:
                f.write("\n".join(new_label_lines))


for i, path in enumerate(dataset_paths):
    copy_dataset(path, f"ds{i}")

print("Объединение завершено")

Датасет: /kaggle/working/people-and-bags-1
Старые классы: ['bag', 'person']
Карта классов: {0: 1, 1: 0}
------------------------------------------------------------
Датасет: /kaggle/working/Bags-1
Старые классы: ['Backpack', 'Bag', 'Cloth-Bag', 'Handbag', 'backpack', 'bad rad', 'bag', 'bag black', 'bag blue', 'bag brown', 'bag cream', 'bag green', 'bag orange', 'bag pink', 'bag rad', 'bag silver', 'bag white', 'bag yellow', 'bolso', 'cable lock', 'cartera', 'compartment', 'doll', 'ecobag', 'handbag', 'handle', 'keychain', 'lock', 'logo', 'mochila', 'person', 'receptacle', 'sash', 'tas', 'waste-backpack', 'zipper']
Карта классов: {0: 1, 1: 1, 3: 1, 4: 1, 6: 1, 24: 1, 30: 0}
------------------------------------------------------------
Датасет: /kaggle/working/bag-detection-1
Старые классы: ['backpack', 'satchel', 'tote bag', 'trolley case']
Карта классов: {0: 1}
------------------------------------------------------------
Датасет: /kaggle/working/Backpack-1
Старые классы: ['Backpack']
Ка

In [6]:
data = {
    "path": str(final_path),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        0: "person",
        1: "bag"
    }
}

with open(final_path / "data.yaml", "w") as f:
    yaml.dump(data, f, allow_unicode=True)

print(" data.yaml создан:")
print(final_path / "data.yaml")

 data.yaml создан:
/kaggle/working/final_people_bag_dataset_v3/data.yaml


In [7]:
for split in ["train", "valid", "test"]:
    label_dir = final_path / split / "labels"

    person_count = 0
    bag_count = 0

    for txt in label_dir.glob("*.txt"):
        with open(txt, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) == 0:
                    continue

                cls = parts[0]

                if cls == "0":
                    person_count += 1
                elif cls == "1":
                    bag_count += 1

    img_count = len(list((final_path / split / "images").glob("*")))
    lbl_count = len(list((final_path / split / "labels").glob("*")))

    print(split)
    print("images:", img_count)
    print("labels:", lbl_count)
    print("person:", person_count)
    print("bag:", bag_count)
    print("-" * 40)

train
images: 9194
labels: 9194
person: 10866
bag: 11489
----------------------------------------
valid
images: 1869
labels: 1869
person: 2913
bag: 2346
----------------------------------------
test
images: 913
labels: 913
person: 1480
bag: 1128
----------------------------------------


In [8]:
model = YOLO(old_model_path)

model.train(
    data="/kaggle/working/final_people_bag_dataset_v3/data.yaml",
    epochs=80,
    imgsz=768,
    batch=8,
    patience=15,
    device=0,
    name="people_bag_detector_v3"
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/final_people_bag_dataset_v3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/leslyhihihi/googlecolab2/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=people_bag_detector_v3, nbs=64, nms=F

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bbf09a351c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [9]:
os.makedirs("/kaggle/working/saved_model", exist_ok=True)

source_model = "/kaggle/working/runs/detect/people_bag_detector_v3/weights/best.pt"
saved_model = "/kaggle/working/saved_model/people_bag_detector_v3_best.pt"

shutil.copy(source_model, saved_model)

print("Финальная модель сохранена:")
print(saved_model)

Финальная модель сохранена:
/kaggle/working/saved_model/people_bag_detector_v3_best.pt


In [10]:
shutil.make_archive(
    "/kaggle/working/people_bag_detector_v3_model",
    "zip",
    "/kaggle/working/saved_model"
)

print("Архив модели создан:")
print("/kaggle/working/people_bag_detector_v3_model.zip")

Архив модели создан:
/kaggle/working/people_bag_detector_v3_model.zip


In [1]:
# путь до фото
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(4).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(5).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(7).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/images (9).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(8).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/images (6).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(25).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/S t r e e t s.jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/2026-05-26 051633.png
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/77.jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(12).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/.jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(10).jpg
/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(16).jpg
/kaggle/in

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os
import glob
import shutil

# очищаем старые результаты
if os.path.exists("/kaggle/working/test_results"):
    shutil.rmtree("/kaggle/working/test_results")

# путь к финальной модели
model_path = "/kaggle/working/saved_model/people_bag_detector_v3_best.pt"

model = YOLO(model_path)

# путь к тестовой фотографии
image_path = "/kaggle/input/datasets/leslyhihihi/test-bags-2-2/тест_новый/(4).jpg"

print("Проверяем изображение:")
print(image_path)

display(Image(filename=image_path))

results = model.predict(
    source=image_path,
    conf=0.25,
    imgsz=768,
    save=True,
    project="/kaggle/working/test_results",
    name="predict",
    exist_ok=True
)

boxes = results[0].boxes

people_count = 0
bag_count = 0

for box in boxes:
    cls = int(box.cls[0].item())
    conf = box.conf[0].item()

    if cls == 0:
        people_count += 1
        print(f"person — уверенность {conf:.2%}")
    elif cls == 1:
        bag_count += 1
        print(f"bag — уверенность {conf:.2%}")

print("Людей найдено:", people_count)
print("Сумок найдено:", bag_count)

if people_count > 0 and bag_count > 0:
    print("На изображении есть человек с сумкой")
elif people_count > 0 and bag_count == 0:
    print("Человек найден, но сумка не найдена")
elif people_count == 0 and bag_count > 0:
    print("Сумка найдена, но человек не найден")
else:
    print("Человек с сумкой не найден")

result_images = glob.glob("/kaggle/working/test_results/predict/*")

if result_images:
    display(Image(filename=result_images[0]))
else:
    print("Результат не найден")